# 03 · Testing (`unittest`, `pytest`, `doctest`) & Code-Quality Tools (Ruff, pylint, mypy)

**Module 07 – Errors, Debugging & Testing** · Estimated time: 60 min

> Updates 📗 *Unit Testing* (pylint + unittest). ⚠️ The bootcamp assumes Anaconda ships pylint and doesn't mention **pytest**, which is now the de-facto standard test runner, or **Ruff**, the modern linter/formatter.

### 🎯 What you will learn
- Why automated tests matter
- Linters: `pylint` (📗) and **Ruff** (modern, very fast)
- `unittest` (standard library, 📗 example)
- ✅ `pytest`: plain `assert`, fixtures, parametrisation, testing exceptions
- `doctest`: tests inside docstrings
- Test coverage, and how it fits in a project

> 💡 `%%writefile name.py` (an IPython magic) saves the cell's contents to a file. `!command` runs a shell command. We use `{sys.executable}` so the shell uses the same Python as this notebook.

In [1]:
import sys
print(sys.executable)

/private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f4ffaf20-c7ab-4c1c-ae64-d5055060b56d/scratchpad/venv315/bin/python


## 1. Why test?
- Catch bugs **before** users do
- Change code confidently (**refactoring** without fear)
- Tests document how code is supposed to behave
- Required in virtually every professional codebase (CI pipelines run them on every commit)

## 2. Linters: catching problems without running the code

### pylint (📗)

In [2]:
%%writefile simple1.py
a = 1
b = 2
print(a)
print(B)

Writing simple1.py


In [3]:
!{sys.executable} -m pylint simple1.py --score=y

************* Module simple1
simple1.py:1:0: C0114: Missing module docstring (missing-module-docstring)
simple1.py:1:0: C0103: Constant name "a" doesn't conform to UPPER_CASE naming style (invalid-name)
simple1.py:2:0: C0103: Constant name "b" doesn't conform to UPPER_CASE naming style (invalid-name)
simple1.py:4:6: E0602: Undefined variable 'B' (undefined-variable)



--------------------------------------------------------------------
Your code has been rated at 0.00/10 (previous run: 10.00/10, -10.00)



pylint found: missing module docstring, constant naming style, and, importantly, an **undefined variable** `B` (a real bug!). Fix and re-run:

In [4]:
%%writefile simple1.py
"""
A very simple script.
"""

def myfunc():
    """
    An extremely simple function.
    """
    first = 1
    second = 2
    print(first)
    print(second)

myfunc()

Overwriting simple1.py


In [5]:
!{sys.executable} -m pylint simple1.py --score=y


--------------------------------------------------------------------
Your code has been rated at 10.00/10 (previous run: 0.00/10, +10.00)



### ✅ Ruff: the modern linter **and** formatter

[Ruff](https://docs.astral.sh/ruff/) is written in Rust, is 10–100× faster than older tools, and replaces flake8, isort, pyupgrade and Black in one tool. Most new projects use it.

In [6]:
%%writefile messy.py
import os, sys
from typing import List
def add( a,b ):
    x=1
    return a+b
values : List[int] = [1,2,3]
if values == None:
    print( "empty" )

Writing messy.py


In [7]:
!{sys.executable} -m ruff check messy.py --select E,F,UP,I --no-cache --output-format concise

messy.py:1:1: E401 [*] Multiple imports on one line
messy.py:1:1: I001 [*] Import block is un-sorted or un-formatted
messy.py:1:8: F401 [*] `os` imported but unused
messy.py:1:12: F401 [*] `sys` imported but unused
messy.py:2:1: UP035 `typing.List` is deprecated, use `list` instead
messy.py:4:5: F841 Local variable `x` is assigned to but never used
messy.py:6:10: UP006 [*] Use `list` instead of `List` for type annotation
messy.py:7:14: E711 Comparison to `None` should be `cond is None`
Found 8 errors.
[*] 5 fixable with `--fix` (2 unsafe fixes available with `--fix --unsafe-fixes`).


In [8]:
!{sys.executable} -m ruff format messy.py --no-cache && cat messy.py

1 file reformatted
import os, sys
from typing import List


def add(a, b):
    x = 1
    return a + b


values: List[int] = [1, 2, 3]
if values == None:
    print("empty")


Ruff flagged unused imports, an unused variable, the old `typing.List`, and `== None`, then reformatted the spacing automatically. Use `ruff check --fix` to auto-fix many issues.

> 💡 Also run a **type checker** (`mypy` or `pyright`, Module 05 · Type Hints). Lint + format + type check + tests = professional quality.

## 3. `unittest` (standard library, 📗)

The code under test:

In [9]:
%%writefile cap.py
def cap_text(text):
    """Return the text with each word capitalized."""
    return text.title()

Writing cap.py


Tests are methods of a `unittest.TestCase` subclass whose names start with `test`:

In [10]:
%%writefile test_cap_unittest.py
import unittest
import cap

class TestCap(unittest.TestCase):

    def test_one_word(self):
        text = "python"
        result = cap.cap_text(text)
        self.assertEqual(result, "Python")

    def test_multiple_words(self):
        text = "monty python"
        result = cap.cap_text(text)
        self.assertEqual(result, "Monty Python")

    def test_with_apostrophes(self):
        text = "monty python's flying circus"
        result = cap.cap_text(text)
        self.assertEqual(result, "Monty Python's Flying Circus")

if __name__ == "__main__":
    unittest.main()

Writing test_cap_unittest.py


In [11]:
!{sys.executable} -m unittest -v test_cap_unittest

test_multiple_words (test_cap_unittest.TestCap.test_multiple_words) ... ok
test_one_word (test_cap_unittest.TestCap.test_one_word) ... ok
test_with_apostrophes (test_cap_unittest.TestCap.test_with_apostrophes) ... FAIL

FAIL: test_with_apostrophes (test_cap_unittest.TestCap.test_with_apostrophes)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/tmp6emth1ca/test_cap_unittest.py", line 19, in test_with_apostrophes
    self.assertEqual(result, "Monty Python's Flying Circus")
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: "Monty Python'S Flying Circus" != "Monty Python's Flying Circus"
- Monty Python'S Flying Circus
?              ^
+ Monty Python's Flying Circus
?              ^


----------------------------------------------------------------------
Ran 3 tests in 0.000s

FAILED (failures=1)


The apostrophe test **fails**: `str.title()` produces `"Python'S"`. The test found a real bug! Let's fix the code with `string.capwords`:

In [12]:
%%writefile cap.py
import string

def cap_text(text):
    """Return the text with each word capitalized."""
    return string.capwords(text)

Overwriting cap.py


In [13]:
!{sys.executable} -m unittest -v test_cap_unittest

test_multiple_words (test_cap_unittest.TestCap.test_multiple_words) ... ok
test_one_word (test_cap_unittest.TestCap.test_one_word) ... ok
test_with_apostrophes (test_cap_unittest.TestCap.test_with_apostrophes) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.000s

OK


### Common `unittest` assertions

| Method | Checks |
|---|---|
| `assertEqual(a, b)` / `assertNotEqual` | `a == b` |
| `assertTrue(x)` / `assertFalse(x)` | truthiness |
| `assertIs(a, b)` / `assertIsNone(x)` | identity |
| `assertIn(a, b)` | `a in b` |
| `assertIsInstance(a, T)` | type |
| `assertAlmostEqual(a, b)` | floats |
| `assertRaises(Err)` | an exception is raised |

`setUp()` / `tearDown()` run before/after **each** test for shared preparation.

## 4. ✅ `pytest`: the modern standard

Install: `pip install pytest`. Tests are **plain functions** using plain **`assert`**. pytest shows rich failure messages.

In [14]:
%%writefile shop.py
from dataclasses import dataclass, field

class OutOfStockError(Exception):
    pass

@dataclass
class Cart:
    items: dict[str, int] = field(default_factory=dict)

    def add(self, name: str, qty: int = 1) -> None:
        if qty <= 0:
            raise ValueError("qty must be positive")
        self.items[name] = self.items.get(name, 0) + qty

    def total(self, prices: dict[str, float]) -> float:
        return round(sum(prices[n] * q for n, q in self.items.items()), 2)

def discount(price: float, percent: float) -> float:
    if not 0 <= percent <= 100:
        raise ValueError("percent must be 0-100")
    return round(price * (1 - percent / 100), 2)

Writing shop.py


In [15]:
%%writefile test_shop.py
import pytest
from shop import Cart, discount

# --- simple tests --------------------------------------------------------
def test_add_item():
    cart = Cart()
    cart.add("pen", 2)
    assert cart.items == {"pen": 2}

def test_add_same_item_twice_accumulates():
    cart = Cart()
    cart.add("pen")
    cart.add("pen", 3)
    assert cart.items["pen"] == 4

# --- testing exceptions ---------------------------------------------------
def test_add_rejects_zero_quantity():
    with pytest.raises(ValueError, match="positive"):
        Cart().add("pen", 0)

# --- fixtures: reusable setup ----------------------------------------------
@pytest.fixture
def prices():
    return {"pen": 10.0, "book": 250.5}

@pytest.fixture
def full_cart():
    cart = Cart()
    cart.add("pen", 2)
    cart.add("book")
    return cart

def test_total(full_cart, prices):
    assert full_cart.total(prices) == 270.5

# --- parametrize: many cases, one test --------------------------------------
@pytest.mark.parametrize("price, percent, expected", [
    (100, 10, 90.0),
    (100, 0, 100.0),
    (100, 100, 0.0),
    (59.99, 15, 50.99),
])
def test_discount(price, percent, expected):
    assert discount(price, percent) == expected

@pytest.mark.parametrize("bad", [-1, 101])
def test_discount_rejects_bad_percent(bad):
    with pytest.raises(ValueError):
        discount(100, bad)

Writing test_shop.py


In [16]:
!{sys.executable} -m pytest test_shop.py -v -p no:cacheprovider --color=no

============================= test session starts ==============================
platform darwin -- Python 3.15.0, pytest-9.1.1, pluggy-1.7.0 -- /private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f4ffaf20-c7ab-4c1c-ae64-d5055060b56d/scratchpad/venv315/bin/python


hypothesis profile 'default'
rootdir: /private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/tmp6emth1ca
plugins: cov-7.1.0, asyncio-1.4.0, hypothesis-6.170.5, mock-3.17.0, anyio-4.15.1, platformdirs-4.13.5
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 10 items                                                             

test_shop.py::test_add_item PASSED                                       [ 10%]
test_shop.py::test_add_same_item_twice_accumulates PASSED                [ 20%]
test_shop.py::test_add_rejects_zero_quantity PASSED                      [ 30%]
test_shop.py::test_total PASSED                                          [ 40%]
test_shop.py::test_discount[100-10-90.0] PASSED                          [ 50%]
test_shop.py::test_discount[100-0-100.0] PASSED                          [ 60%]
test_shop.py::test_discount[100-100-0.0] PASSED                          [ 70%]
test_shop.py::test_discount[

### What a failure looks like

In [17]:
%%writefile test_fail_demo.py
def test_broken_math():
    expected = [1, 2, 3]
    actual = [1, 2, 4]
    assert actual == expected

Writing test_fail_demo.py


In [18]:
!{sys.executable} -m pytest test_fail_demo.py -q -p no:cacheprovider --color=no

F                                                                        [100%]
=================================== FAILURES ===================================
_______________________________ test_broken_math _______________________________

    def test_broken_math():
        expected = [1, 2, 3]
        actual = [1, 2, 4]
>       assert actual == expected
E       assert [1, 2, 4] == [1, 2, 3]
E         
E         At index 2 diff: 4 != 3
E         Use -v to get more diff

test_fail_demo.py:4: AssertionError


=========================== short test summary info ============================
FAILED test_fail_demo.py::test_broken_math - assert [1, 2, 4] == [1, 2, 3]
1 failed in 0.06s


pytest shows *exactly* which element differs. That's why people love it.

### Test coverage

In [19]:
!{sys.executable} -m pytest test_shop.py -q -p no:cacheprovider --color=no --cov=shop --cov-report=term-missing

..........                                                               [100%]
================================ tests coverage ================================
_______________ coverage: platform darwin, python 3.15.0-final-0 _______________

Name      Stmts   Miss  Cover   Missing
---------------------------------------
shop.py      16      0   100%
---------------------------------------
TOTAL        16      0   100%


10 passed in 0.06s


## 5. `doctest`: examples in docstrings that are also tests

In [20]:
%%writefile mathx.py
def factorial(n):
    """Return n! for a non-negative integer n.

    >>> factorial(5)
    120
    >>> factorial(0)
    1
    >>> factorial(-1)
    Traceback (most recent call last):
        ...
    ValueError: n must be >= 0
    """
    if n < 0:
        raise ValueError("n must be >= 0")
    return 1 if n <= 1 else n * factorial(n - 1)

Writing mathx.py


In [21]:
!{sys.executable} -m doctest -v mathx.py | tail -3

3 tests in 2 items.
3 passed.
Test passed.


## 6. Testing in a real project

```
my_project/
├── pyproject.toml          ← project config (Module 08); [tool.pytest.ini_options], [tool.ruff]
├── src/my_package/
│   ├── __init__.py
│   └── shop.py
└── tests/
    ├── test_shop.py
    └── conftest.py         ← shared fixtures
```
Run everything with `pytest` from the project root. In CI (e.g. GitHub Actions), every push runs `ruff check`, `mypy` and `pytest`.

**Good tests are:** fast, independent, repeatable, and focused on one behaviour each. Name them after the behaviour: `test_add_rejects_zero_quantity`.

> ➡️ Fixtures in depth, mocking, Hypothesis and async tests: **lesson 04**. CI, pre-commit and Docker: **lesson 05**.

## 📝 Summary
- Linters (Ruff, pylint) find bugs and style issues without running code; Ruff also formats.
- `unittest`: classes + `self.assert*` methods (built in).
- ✅ `pytest`: functions + `assert`, fixtures, `parametrize`, `pytest.raises`.
- `doctest` keeps docstring examples correct.
- Measure coverage with `pytest --cov`.

## 🏋️ Practice
1. Write pytest tests for `is_palindrome(s)` covering: simple palindrome, mixed case, spaces, empty string, non-palindrome. Use `parametrize`.
2. Write a test that checks `int("abc")` raises `ValueError`.
3. Add a doctest to a function `celsius_to_f(c)`.

<details><summary>Show solutions</summary>

```python
# test_palindrome.py: in a real project, is_palindrome lives in palindrome.py
# and the test file does `from palindrome import is_palindrome`
import pytest

def is_palindrome(s):
    t = "".join(s.lower().split())
    return t == t[::-1]

@pytest.mark.parametrize("text, expected", [
    ("racecar", True), ("RaceCar", True), ("never odd or even", True),
    ("", True), ("python", False),
])
def test_is_palindrome(text, expected):
    assert is_palindrome(text) is expected

def test_int_raises():
    with pytest.raises(ValueError):
        int("abc")

def celsius_to_f(c):
    """
    >>> celsius_to_f(100)
    212.0
    """
    return c * 9 / 5 + 32
```
</details>